# 🤖 Smart Auto-Detecting Chatbot
### Understands BOTH plain questions AND MCQ format — responds accordingly
---
```
User types anything
        ↓
  FORMAT DETECTOR  (NLP classifier)
   /            \
Plain Question   MCQ Format
   ↓                  ↓
TF-IDF retrieval   Parse options A/B/C/D
Find best answer   Naive Bayes scores each option
Return explanation Identify correct + explain why
```
**MCQ formats the model understands:**
```
Type 1 — Options on same line:
  What is erosion? A) wearing away B) heating up C) freezing D) expanding

Type 2 — Options on separate lines:
  What is erosion?
  A) wearing away of earth materials
  B) heating up of rocks
  C) freezing of water
  D) expanding of minerals

Type 3 — Numbered options:
  What is erosion?
  1) wearing away  2) heating up  3) freezing  4) expanding
```

## 📦 Step 0 — Install & Import

In [ ]:
!pip install nltk scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import re
import io
import random
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

for r in ['stopwords','punkt','wordnet','punkt_tab']:
    nltk.download(r, quiet=True)

print('✅ All dependencies loaded!')

## 📂 Step 1 — Load Dataset & Build Knowledge Base

In [ ]:
# ── Jupyter ──
df = pd.read_csv('test-uq-00001.csv')

# ── Google Colab: uncomment below ──
# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded:
#     df = pd.read_csv(io.StringIO(open(filename).read()))

# One row per unique question = knowledge base
kb = (
    df.groupby('question')
      .agg(reference_answer=('reference_answer','first'))
      .reset_index()
)

# Label mapping: 0=Correct 1=Contradictory 2=Partially Correct 3=Incorrect
LABEL_MAP = {0:'Correct', 1:'Contradictory', 2:'Partially Correct', 3:'Incorrect'}
df['label_str'] = df['label'].map(LABEL_MAP)

print(f'✅ Knowledge base: {len(kb)} Q&A pairs loaded')
print(f'   Student answers for NB training: {len(df)}')

## 🧹 Step 2 — Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    """Lowercase → remove punctuation → tokenize → lemmatize → remove stopwords."""
    if not text or pd.isna(text):
        return ''
    text   = str(text).lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)
    text   = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    return ' '.join(
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words and len(t) > 2
    )

kb['question_clean'] = kb['question'].apply(preprocess)
kb['answer_clean']   = kb['reference_answer'].apply(preprocess)
df['answer_clean']   = df['student_answer'].apply(preprocess)

print('✅ Preprocessing complete.')

## 🔍 Step 3 — TF-IDF Retrieval Engine

In [ ]:
# Indexes all KB questions for similarity search
retrieval_tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True)
kb_vectors      = retrieval_tfidf.fit_transform(kb['question_clean'])

print(f'✅ Retrieval engine: {kb_vectors.shape[0]} questions indexed')

## 🧠 Step 4 — Naive Bayes Option Scorer

Trained on 733 labelled student answers — used to score each MCQ option and identify which one is most 'Correct'.

In [ ]:
nb_tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=6000, sublinear_tf=True)
X_nb     = nb_tfidf.fit_transform(df['answer_clean'].fillna(''))
y_nb     = df['label_str']

nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_nb, y_nb)

# Accuracy check
X_tr, X_te, y_tr, y_te = train_test_split(X_nb, y_nb, test_size=0.2, random_state=42)
nb_q = MultinomialNB(alpha=0.5)
nb_q.fit(X_tr, y_tr)
acc = (nb_q.predict(X_te) == y_te).mean()
print(f'✅ Naive Bayes scorer trained — test accuracy: {acc:.1%}')

def score_option(option_text: str) -> dict:
    """Returns NB label + probability for a single option text."""
    clean  = preprocess(option_text)
    vec    = nb_tfidf.transform([clean])
    label  = nb_model.predict(vec)[0]
    probs  = nb_model.predict_proba(vec)[0]
    conf   = float(probs.max())
    prob_d = dict(zip(nb_model.classes_, [round(float(p),3) for p in probs]))
    return {'label': label, 'confidence': round(conf,3), 'probs': prob_d}

## 🕵️ Step 5 — Format Detector
The core intelligence — detects whether input is a plain question or an MCQ, and parses the MCQ options if present.

In [ ]:
# Patterns that signal MCQ format
MCQ_PATTERNS = [
    # A) option  B) option  (letter + bracket/dot)
    r'\b[AaBbCcDd][\)\.]\s*\S+',
    # 1) option  2) option  (number + bracket)
    r'\b[1234][\)\.]\s*\S+',
    # (A) option format
    r'\([AaBbCcDd]\)\s*\S+',
    # option1 / option2 / option3 style
    r'\w+\s*/\s*\w+\s*/\s*\w+',
]

def is_mcq(text: str) -> bool:
    """Returns True if the input looks like an MCQ."""
    for pattern in MCQ_PATTERNS:
        matches = re.findall(pattern, text)
        if len(matches) >= 2:   # need at least 2 option markers
            return True
    return False

def parse_mcq(text: str) -> dict:
    """
    Parses MCQ input into question stem + options dict.

    Handles all these formats:
      • A) opt  B) opt  C) opt  D) opt   (same line)
      • A) opt\nB) opt\nC) opt\nD) opt   (separate lines)
      • 1) opt  2) opt  3) opt  4) opt   (numbered)
      • (A) opt  (B) opt                 (parentheses)
    """
    text = text.strip()

    # Normalise: convert numbered options to lettered
    text = re.sub(r'\b1[\)\.]', 'A)', text)
    text = re.sub(r'\b2[\)\.]', 'B)', text)
    text = re.sub(r'\b3[\)\.]', 'C)', text)
    text = re.sub(r'\b4[\)\.]', 'D)', text)
    # Normalise parenthesis style: (A) → A)
    text = re.sub(r'\(([AaBbCcDd])\)', lambda m: m.group(1).upper()+')', text)
    # Normalise A. → A)
    text = re.sub(r'\b([AaBbCcDd])\.\s', lambda m: m.group(1).upper()+') ', text)

    # Split on option markers to get positions
    option_pattern = r'\b([ABCD])\)'
    splits = re.split(option_pattern, text, flags=re.IGNORECASE)

    if len(splits) < 3:
        return {'question': text, 'options': {}}

    # Everything before the first option marker = the question stem
    question_stem = splits[0].strip()

    # Rebuild options dict
    options = {}
    for i in range(1, len(splits)-1, 2):
        letter = splits[i].upper()
        value  = splits[i+1].strip().rstrip(',;') if i+1 < len(splits) else ''
        if letter in 'ABCD' and value:
            options[letter] = value

    return {'question': question_stem, 'options': options}


# ── Quick sanity tests ──
tests = [
    'What is erosion? A) wearing away B) heating up C) freezing D) expanding',
    'What is a mineral?\nA) made of many substances\nB) a single earth substance\nC) a type of rock\nD) a liquid',
    'What is erosion? 1) wearing away 2) heating up 3) freezing 4) expanding',
    'What is the difference between a rock and a mineral?',  # plain question
]
print('Format detection tests:')
for t in tests:
    fmt = 'MCQ' if is_mcq(t) else 'Plain question'
    print(f'  [{fmt}]  {t[:65]}')
    if is_mcq(t):
        parsed = parse_mcq(t)
        print(f'           Q: {parsed["question"][:60]}')
        print(f'           Options: {list(parsed["options"].keys())}')

## 💡 Step 6 — MCQ Answering Logic
When MCQ is detected, the model:
1. Retrieves the reference answer from the KB using the question stem
2. Scores each option against the reference using cosine similarity
3. Uses Naive Bayes to classify each option as Correct/Partially Correct/Incorrect
4. Combines both signals to identify the best answer

In [ ]:
def answer_mcq(parsed: dict) -> None:
    """
    Given a parsed MCQ dict, find the correct option and explain why.
    
    Scoring uses two signals:
      Signal 1 — TF-IDF cosine similarity: how similar is each option
                 to the reference answer from the knowledge base?
      Signal 2 — Naive Bayes label: is the option text classified
                 as Correct / Partially Correct / Incorrect?
    Combined → pick the option with highest cosine similarity to reference.
    """
    question_stem = parsed['question']
    options       = parsed['options']

    if len(options) < 2:
        print('⚠️  Could not parse enough options. Please check the MCQ format.')
        return

    # ── 1. Retrieve reference answer from KB ──
    q_clean   = preprocess(question_stem)
    q_vec     = retrieval_tfidf.transform([q_clean])
    sims      = cosine_similarity(q_vec, kb_vectors).flatten()
    best_idx  = sims.argmax()
    q_score   = sims[best_idx]
    ref_answer = kb.iloc[best_idx]['reference_answer']
    matched_q  = kb.iloc[best_idx]['question']

    # ── 2. Score each option against the reference answer ──
    ref_clean  = preprocess(ref_answer)
    ref_vec    = retrieval_tfidf.transform([ref_clean])

    option_scores = {}
    for letter, opt_text in options.items():
        opt_clean  = preprocess(opt_text)
        opt_vec    = retrieval_tfidf.transform([opt_clean])
        cos_sim    = cosine_similarity(opt_vec, ref_vec).flatten()[0]
        nb_result  = score_option(opt_text)
        option_scores[letter] = {
            'text'      : opt_text,
            'cos_sim'   : round(float(cos_sim), 4),
            'nb_label'  : nb_result['label'],
            'nb_conf'   : nb_result['confidence'],
            'nb_probs'  : nb_result['probs'],
        }

    # ── 3. Pick best answer: highest cosine similarity to reference ──
    best_letter = max(option_scores, key=lambda l: option_scores[l]['cos_sim'])

    # ── 4. Print formatted response ──
    sep = '─' * 62
    print(f'\n{sep}')
    print(f'📋 MCQ DETECTED')
    print(f'{sep}')
    print(f'❓ Question : {question_stem}')
    if q_score > 0.1 and matched_q.lower() != question_stem.lower():
        print(f'🔍 Matched  : {matched_q[:72]}' + ('...' if len(matched_q)>72 else ''))
    print(f'{sep}')
    print(f'{'Option':<6}  {'Text':<38}  {'Similarity':<11}  NB Label')
    print(f'{'─'*6}  {'─'*38}  {'─'*11}  {'─'*16}')

    for letter in sorted(option_scores):
        s    = option_scores[letter]
        star = '  ✅ ANSWER' if letter == best_letter else ''
        disp_text = s['text'][:36] + '..' if len(s['text']) > 36 else s['text']
        print(f'  {letter}     {disp_text:<38}  {s["cos_sim"]:.4f}       {s["nb_label"]}{star}')

    print(f'{sep}')
    print(f'✅ Correct Answer : ({best_letter}) {options[best_letter]}')
    print(f'\n📖 Explanation:')
    print(f'   {ref_answer}')
    print(f'{sep}')

print('✅ MCQ answering engine ready.')

## 💬 Step 7 — Plain Q&A Answering Logic

In [ ]:
INTENT_RULES = [
    ('explain',  r'\b(explain|why|how does|how do|reason|cause|because)\b'),
    ('compare',  r'\b(difference|compare|similar|versus|between|contrast)\b'),
    ('define',   r'\b(what is|what are|define|meaning)\b'),
    ('evidence', r'\b(evidence|proof|show|indicate|data|experiment)\b'),
    ('predict',  r'\b(what happen|what will|predict|expect|would happen)\b'),
]
INTENT_OPENERS = {
    'explain' : 'Here is the explanation:',
    'compare' : 'Here is the comparison:',
    'define'  : 'Here is the definition:',
    'evidence': 'Based on the evidence:',
    'predict' : 'Based on what we know:',
    'general' : 'Here is what I found:',
}
FALLBACKS = [
    "I don't have enough information on that. Type 'topics' to see what I can answer.",
    "That's outside my knowledge base. Try asking about rocks, minerals, erosion, circuits, or sound.",
]
CONFIDENCE_THRESHOLD = 0.12

def detect_intent(q: str) -> str:
    q = q.lower()
    for intent, pattern in INTENT_RULES:
        if re.search(pattern, q):
            return intent
    return 'general'

def answer_plain(user_question: str) -> None:
    """Answer a plain text question using TF-IDF retrieval."""
    q_clean  = preprocess(user_question)
    q_vec    = retrieval_tfidf.transform([q_clean])
    sims     = cosine_similarity(q_vec, kb_vectors).flatten()
    best_idx = sims.argmax()
    score    = sims[best_idx]
    intent   = detect_intent(user_question)
    opener   = INTENT_OPENERS.get(intent, INTENT_OPENERS['general'])
    sep      = '─' * 62

    print(f'\n{sep}')
    print(f'💬 PLAIN QUESTION DETECTED')
    print(f'{sep}')
    print(f'🧑 You    : {user_question}')
    print(f'🎯 Intent : {intent.upper()}  |  Confidence: {score:.1%}')
    print(f'{sep}')

    if score < CONFIDENCE_THRESHOLD:
        print(f'🤖 Bot    : {random.choice(FALLBACKS)}')
    else:
        matched = kb.iloc[best_idx]['question']
        answer  = kb.iloc[best_idx]['reference_answer']
        if matched.lower() != user_question.lower():
            disp = matched[:72]+('...' if len(matched)>72 else '')
            print(f'🔍 Matched: {disp}')
            print()
        print(f'🤖 Bot    : {opener}')
        print(f'\n   {answer}')
    print(f'{sep}')

print('✅ Plain Q&A engine ready.')

## 🧪 Step 8 — Test Both Formats

In [ ]:
print('━'*62)
print('TEST 1 — Plain question')
print('━'*62)
answer_plain('What is the difference between a rock and a mineral?')

In [ ]:
print('━'*62)
print('TEST 2 — MCQ on same line')
print('━'*62)
mcq_input = 'What happens to earth materials during erosion? A) They are worn away and moved B) They heat up and expand C) They freeze and crack D) They dissolve in water'
parsed = parse_mcq(mcq_input)
answer_mcq(parsed)

In [ ]:
print('━'*62)
print('TEST 3 — MCQ with options on separate lines')
print('━'*62)
mcq_multiline = """What is the difference between a rock and a mineral?
A) A rock is made of one substance, a mineral is made of many
B) A rock is made of many minerals, a mineral is a single substance
C) They are the same thing
D) Minerals are bigger than rocks"""
parsed2 = parse_mcq(mcq_multiline)
answer_mcq(parsed2)

In [ ]:
print('━'*62)
print('TEST 4 — MCQ with numbered options')
print('━'*62)
mcq_numbered = 'What happens to the volume of sound if you pluck a rubber band harder? 1) It stays the same 2) It gets louder 3) It gets softer 4) It disappears'
parsed3 = parse_mcq(mcq_numbered)
answer_mcq(parsed3)

In [ ]:
print('━'*62)
print('TEST 5 — Plain question (paraphrased)')
print('━'*62)
answer_plain('How does erosion change the land?')

## 🗨️ Step 9 — Full Interactive Chatbot
**Just type naturally — the bot figures out the format automatically.**

In [ ]:
def print_banner():
    print('\n' + '═'*62)
    print('  🤖  SMART AUTO-DETECTING SCIENCE CHATBOT')
    print('═'*62)
    print('  Just type — the bot detects the format automatically!')
    print()
    print('  Plain question examples:')
    print('    What is erosion?')
    print('    How does a parallel circuit work?')
    print('    Why does a tighter string make a higher pitch?')
    print()
    print('  MCQ examples (any of these formats work):')
    print('    What is erosion? A) wearing away B) heating up C) freezing D) expanding')
    print('    What is erosion?')
    print('    A) wearing away')
    print('    B) heating up ...')
    print()
    print("  Type 'topics' to see all topics  |  'quit' to exit")
    print('═'*62 + '\n')

print_banner()

while True:
    # Collect input — support multiline MCQ entry
    print()
    first_line = input('🧑 You: ').strip()

    if not first_line:
        continue

    cmd = first_line.lower()

    # ── built-in commands ──
    if cmd == 'quit':
        print('\n🤖 Bot: Goodbye! Keep learning! 👋')
        break

    if cmd == 'topics':
        print('\n🤖 Bot: Topics I can answer:')
        for i, q in enumerate(kb['question'].tolist(), 1):
            disp = q[:82]+('...' if len(q)>82 else '')
            print(f'  {i:2}. {disp}')
        continue

    if cmd == 'help':
        print_banner()
        continue

    # ── collect extra lines if MCQ options are on separate lines ──
    # If the first line has no option markers yet, peek for more
    full_input = first_line
    if not is_mcq(first_line):
        # Check if user might be entering multiline MCQ
        # Give them up to 4 more lines; stop early if blank
        extra_lines = []
        for _ in range(4):
            try:
                nxt = input('         ').strip()
            except EOFError:
                break
            if not nxt:
                break
            extra_lines.append(nxt)
            # If we've now collected enough option markers, stop collecting
            combined = full_input + '\n' + '\n'.join(extra_lines)
            if is_mcq(combined):
                break
        if extra_lines:
            full_input = full_input + '\n' + '\n'.join(extra_lines)

    # ── route to correct handler ──
    if is_mcq(full_input):
        parsed = parse_mcq(full_input)
        answer_mcq(parsed)
    else:
        answer_plain(full_input)